In [ ]:
!pip install -q streamlit pyjwt bcrypt pyngrok plotly streamlit-option-menu email-validator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 59.0 MB/s eta 0:00:00


In [ ]:
import os
import re
import sqlite3
import random
import smtplib
import bcrypt
import jwt
import datetime
import streamlit as st
import plotly.graph_objects as go

from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email_validator import validate_email, EmailNotValidError
from streamlit_option_menu import option_menu

from google.colab import userdata

# ==============================
# Load Secrets from Google Colab
# ==============================

JWT_SECRET = userdata.get("JWT_SECRET")
NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
EMAIL_ADDRESS = userdata.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = userdata.get("EMAIL_PASSWORD")

# ==============================
# Hardcoded Admin Credentials
# (Required by Milestone PDF)
# ==============================

ADMIN_EMAIL = "admin@infosys.com"
ADMIN_PASSWORD = "Admin@123"

print("✅ All imports loaded successfully.")
print("✅ Colab Secrets loaded successfully.")

2026-07-08 11:18:39.409 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


✅ All imports loaded successfully.
✅ Colab Secrets loaded successfully.


In [ ]:
# ==========================================================
# DATABASE + HELPER FUNCTIONS
# ==========================================================

DB_NAME = "infosys_portal.db"


def get_db():
    return sqlite3.connect(DB_NAME, check_same_thread=False)


# ----------------------------------------------------------
# Create Users Table
# ----------------------------------------------------------

with get_db() as conn:

    conn.execute("""
    CREATE TABLE IF NOT EXISTS users(

        id INTEGER PRIMARY KEY AUTOINCREMENT,

        username TEXT UNIQUE NOT NULL,

        email TEXT UNIQUE NOT NULL,

        password_hash TEXT NOT NULL,

        security_question TEXT NOT NULL,

        security_answer_hash TEXT NOT NULL

    )
    """)

print("✅ Database Ready")


# ==========================================================
# PASSWORD HASHING
# ==========================================================

def hash_password(password):
    return bcrypt.hashpw(
        password.encode(),
        bcrypt.gensalt()
    ).decode()


def verify_password(password, hashed):

    return bcrypt.checkpw(
        password.encode(),
        hashed.encode()
    )


# ==========================================================
# EMAIL VALIDATION
# ==========================================================

def is_valid_email(email):

    try:
        validate_email(email)

        pattern = r'^[A-Za-z]{2,}@[A-Za-z]{2,}\.[A-Za-z]{2,}$'

        return re.match(pattern, email) is not None

    except EmailNotValidError:

        return False


# ==========================================================
# PASSWORD VALIDATION
# ==========================================================

def is_valid_password(password):

    if len(password) < 8:
        return False

    if not re.search(r"[A-Z]", password):
        return False

    if not re.search(r"[a-z]", password):
        return False

    if not re.search(r"\d", password):
        return False

    if not re.search(r"[!@#$%^&*(),.?\":{}|<>]", password):
        return False

    return True


# ==========================================================
# OTP GENERATOR
# ==========================================================

def generate_otp():

    return str(random.randint(100000, 999999))


# ==========================================================
# EMAIL SENDER
# ==========================================================

def send_otp(receiver_email, otp):

    try:

        subject = "Infosys Portal OTP Verification"

        body = f"""

Hello,

Your OTP for resetting the password is:

{otp}

This OTP is valid for only 5 minutes.

Regards,
Infosys Portal
"""

        message = MIMEMultipart()

        message["From"] = EMAIL_ADDRESS
        message["To"] = receiver_email
        message["Subject"] = subject

        message.attach(MIMEText(body, "plain"))

        server = smtplib.SMTP("smtp.gmail.com", 587)

        server.starttls()

        server.login(
            EMAIL_ADDRESS,
            EMAIL_PASSWORD
        )

        server.sendmail(
            EMAIL_ADDRESS,
            receiver_email,
            message.as_string()
        )

        server.quit()

        return True

    except Exception as e:

        print(e)

        return False


print("✅ Helper Functions Loaded")

✅ Database Ready
✅ Helper Functions Loaded


In [ ]:
# ==========================================================
# JWT AUTHENTICATION
# ==========================================================

JWT_ALGORITHM = "HS256"
JWT_EXPIRE_HOURS = 2


def create_jwt(email):
    """
    Create JWT token after successful login.
    """

    payload = {
        "email": email,
        "exp": datetime.datetime.utcnow()
        + datetime.timedelta(hours=JWT_EXPIRE_HOURS)
    }

    token = jwt.encode(
        payload,
        JWT_SECRET,
        algorithm=JWT_ALGORITHM
    )

    return token


def verify_jwt(token):
    """
    Verify JWT token.
    Returns payload if valid else None.
    """

    try:

        payload = jwt.decode(
            token,
            JWT_SECRET,
            algorithms=[JWT_ALGORITHM]
        )

        return payload

    except jwt.ExpiredSignatureError:

        return None

    except jwt.InvalidTokenError:

        return None


# ==========================================================
# SESSION STATE INITIALIZATION
# ==========================================================

DEFAULT_SESSION = {

    "token": None,

    "page": "Login",

    "otp": None,

    "otp_email": None,

    "otp_verified": False,

    "reset_email": None,

    "reset_mode": None,

    "security_question": None

}


for key, value in DEFAULT_SESSION.items():

    if key not in st.session_state:

        st.session_state[key] = value


# ==========================================================
# SESSION HELPERS
# ==========================================================

def navigate(page):

    st.session_state.page = page

    st.rerun()


def login_user(email):

    token = create_jwt(email)

    st.session_state.token = token

    navigate("Dashboard")


def logout_user():

    st.session_state.token = None

    st.session_state.page = "Login"

    st.session_state.otp = None

    st.session_state.otp_email = None

    st.session_state.otp_verified = False

    st.session_state.reset_email = None

    st.session_state.reset_mode = None

    st.session_state.security_question = None

    st.rerun()


def current_user():

    if st.session_state.token is None:
        return None

    payload = verify_jwt(st.session_state.token)

    if payload is None:
        logout_user()

    return payload


print("✅ JWT Authentication Ready")
print("✅ Session Management Ready")

2026-07-08 11:18:47.068 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 11:18:47.069 Session state does not function when running a script without `streamlit run`
2026-07-08 11:18:47.070 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 11:18:47.071 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 11:18:47.072 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 11:18:47.075 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 11:18:47.075 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 11:18:47.076 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 11:18

✅ JWT Authentication Ready
✅ Session Management Ready


In [ ]:
%%writefile app.py
import os
import re
import sqlite3
import random
import smtplib
import datetime
import bcrypt
import jwt
import streamlit as st
import plotly.graph_objects as go

from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email_validator import validate_email, EmailNotValidError
from streamlit_option_menu import option_menu

# =====================================================
# PAGE CONFIG
# =====================================================

st.set_page_config(
    page_title="Infosys Portal",
    page_icon="⚡",
    layout="wide",
    initial_sidebar_state="expanded"
)

# =====================================================
# SECRETS
# =====================================================

JWT_SECRET = os.environ.get("JWT_SECRET")
EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD")

ADMIN_EMAIL = "admin@infosys.com"
ADMIN_PASSWORD = "Admin@123"

# =====================================================
# COLORS
# =====================================================

PRIMARY = "#005BAC"
SECONDARY = "#0078D4"
ACCENT = "#00A3FF"
BG = "#F4F8FB"
CARD = "#FFFFFF"
TEXT = "#1E293B"
SUCCESS = "#16A34A"
DANGER = "#DC2626"

# =====================================================
# DATABASE
# =====================================================

DB_NAME = "infosys_portal.db"

def get_db():
    return sqlite3.connect(DB_NAME, check_same_thread=False)

# =====================================================
# CSS
# =====================================================

st.markdown(f"""
<style>

.stApp{{
background:{BG};
}}

section[data-testid="stSidebar"]{{
background:#EAF4FF;
}}

div[data-testid="stButton"] button{{
background:{PRIMARY};
color:white;
border-radius:10px;
font-weight:bold;
height:45px;
border:none;
}}

div[data-testid="stButton"] button:hover{{
background:{SECONDARY};
}}

input{{
border-radius:8px;
}}

.card{{
background:white;
padding:25px;
border-radius:15px;
box-shadow:0px 3px 10px rgba(0,0,0,.15);
}}

</style>
""",unsafe_allow_html=True)

print("Infosys Portal Started")


def hash_password(password):
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()

def verify_password(password, hashed):
    return bcrypt.checkpw(password.encode(), hashed.encode())

def is_valid_email(email):
    try:
        validate_email(email)
        pattern = r'^[A-Za-z]{2,}[A-Za-z0-9._%+-]*@[A-Za-z]{2,}\.[A-Za-z]{2,}$'
        return re.match(pattern, email) is not None
    except EmailNotValidError:
        return False

def is_valid_password(password):
    if len(password) < 8:
        return False
    if not re.search(r"[A-Z]", password):
        return False
    if not re.search(r"[a-z]", password):
        return False
    if not re.search(r"\d", password):
        return False
    if not re.search(r'[!@#$%^&*(),.?":{}|<>]', password):
        return False
    return True

PASSWORD_HINT = "Min 8 chars, 1 uppercase, 1 lowercase, 1 number, 1 special symbol."

def generate_otp():
    return str(random.randint(100000, 999999))

def send_otp(receiver_email, otp):
    try:
        subject = "Infosys Portal — Your OTP Verification Code"

        # Plain-text fallback (for email clients that don't render HTML)
        text_body = f"""Hello,

Your OTP for resetting your Infosys Portal password is: {otp}

This OTP is valid for 5 minutes. If you did not request this, please ignore this email.

Regards,
Infosys Portal Team
"""

        # Attractive HTML version
        html_body = f"""
        <html>
          <body style="margin:0;padding:0;background-color:#F4F8FB;font-family:'Segoe UI',Arial,sans-serif;">
            <table role="presentation" width="100%" cellpadding="0" cellspacing="0" style="background-color:#F4F8FB;padding:40px 0;">
              <tr>
                <td align="center">
                  <table role="presentation" width="480" cellpadding="0" cellspacing="0"
                         style="background:#FFFFFF;border-radius:16px;overflow:hidden;box-shadow:0 4px 18px rgba(0,0,0,0.08);">

                    <!-- Header -->
                    <tr>
                      <td style="background:#005BAC;padding:28px 32px;text-align:center;">
                        <div style="font-size:32px;line-height:1;">⚡</div>
                        <div style="color:#FFFFFF;font-size:20px;font-weight:700;margin-top:6px;">
                          Infosys Portal
                        </div>
                        <div style="color:#CBD5E1;font-size:12px;margin-top:2px;">
                          Intelligent Analytics Portal
                        </div>
                      </td>
                    </tr>

                    <!-- Body -->
                    <tr>
                      <td style="padding:36px 32px 8px 32px;text-align:center;">
                        <div style="color:#1E293B;font-size:17px;font-weight:600;margin-bottom:6px;">
                          Password Reset Verification
                        </div>
                        <div style="color:#64748B;font-size:14px;line-height:1.5;">
                          Use the one-time code below to verify your identity and reset your password.
                          This code will expire in <strong>5 minutes</strong>.
                        </div>
                      </td>
                    </tr>

                    <!-- OTP box -->
                    <tr>
                      <td style="padding:24px 32px;text-align:center;">
                        <div style="display:inline-block;background:#EAF4FF;border:2px dashed #00A3FF;
                                    border-radius:12px;padding:16px 36px;">
                          <span style="font-size:32px;font-weight:700;letter-spacing:10px;color:#005BAC;">
                            {otp}
                          </span>
                        </div>
                      </td>
                    </tr>

                    <!-- Footer note -->
                    <tr>
                      <td style="padding:8px 32px 32px 32px;text-align:center;">
                        <div style="color:#94A3B8;font-size:12px;line-height:1.6;">
                          If you didn't request this code, you can safely ignore this email —
                          your account is still secure.
                        </div>
                      </td>
                    </tr>

                    <!-- Bottom bar -->
                    <tr>
                      <td style="background:#F1F5F9;padding:16px 32px;text-align:center;">
                        <div style="color:#94A3B8;font-size:11px;">
                          © Infosys Springboard · Infosys Portal
                        </div>
                      </td>
                    </tr>

                  </table>
                </td>
              </tr>
            </table>
          </body>
        </html>
        """

        message = MIMEMultipart("alternative")
        message["From"] = EMAIL_ADDRESS
        message["To"] = receiver_email
        message["Subject"] = subject

        # Attach plain text first, HTML second — email clients render the
        # last part they support, so HTML is preferred when available.
        message.attach(MIMEText(text_body, "plain"))
        message.attach(MIMEText(html_body, "html"))

        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.sendmail(EMAIL_ADDRESS, receiver_email, message.as_string())
        server.quit()
        return True

    except Exception as e:
        print(e)
        return False

JWT_ALGORITHM = "HS256"
JWT_EXPIRE_HOURS = 2

def create_jwt(email):
    payload = {
        "email": email,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=JWT_EXPIRE_HOURS)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALGORITHM)

def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=[JWT_ALGORITHM])
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None

with get_db() as conn:
    conn.execute("""
    CREATE TABLE IF NOT EXISTS users(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        email TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL,
        security_question TEXT NOT NULL,
        security_answer_hash TEXT NOT NULL
    )
    """)

DEFAULT_SESSION = {
    "token": None, "page": "Login", "is_admin": False,
    "otp": None, "otp_email": None, "otp_sent_at": None,
    "reset_email": None, "reset_mode": None, "security_question": None
}
for key, value in DEFAULT_SESSION.items():
    if key not in st.session_state:
        st.session_state[key] = value

def navigate(page):
    st.session_state.page = page
    st.rerun()

def login_user(email, is_admin=False):
    st.session_state.token = create_jwt(email)
    st.session_state.is_admin = is_admin
    navigate("Dashboard")

def logout_user():
    for key, value in DEFAULT_SESSION.items():
        st.session_state[key] = value
    st.rerun()

def current_user():
    if st.session_state.token is None:
        return None
    payload = verify_jwt(st.session_state.token)
    if payload is None:
        logout_user()
    return payload

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="margin:0;color:{TEXT};">Infosys Portal</h1>
        <p style="color:#64748b;font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;">
        <span style="font-size:1.1rem;font-weight:700;color:{TEXT};">{title}</span>
    </div>
    """, unsafe_allow_html=True)

# =====================================================
# ROUTING
# =====================================================

if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.4, 1])
    with mid:
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
                if not email or not pwd:
                    st.error("⚠️ Please enter both email and password.")
                elif email == ADMIN_EMAIL and pwd == ADMIN_PASSWORD:
                    login_user(email, is_admin=True)
                else:
                    with get_db() as c:
                        r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
                    if r and verify_password(pwd, r[0]):
                        login_user(email, is_admin=False)
                    else:
                        st.error("❌ Invalid credentials.")
            if col_c.button("Create Account", use_container_width=True):
                navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True):
                navigate("Forgot")

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", [
                "What is your pet name?",
                "What is your mother's maiden name?",
                "What is your favourite city?"
            ])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.caption(PASSWORD_HINT)
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                if not uname or not email or not pwd or not confirm_pwd or not sa:
                    st.error("⚠️ Please fill all fields.")
                elif not is_valid_email(email):
                    st.error("❌ Please enter a valid email address (e.g. ab@cd.ef).")
                elif not is_valid_password(pwd):
                    st.error(f"❌ Weak password. {PASSWORD_HINT}")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute(
                                "INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                                (uname, email, hash_password(pwd), sq, hash_password(sa.lower().strip()))
                            )
                        st.success("✅ Account created! Please log in.")
                        st.session_state.page = "Login"
                        st.rerun()
                    except sqlite3.IntegrityError:
                        st.error("❌ Username or Email already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True):
                navigate("Login")

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            if not st.session_state.reset_email:
                email = st.text_input("Registered email address", placeholder="you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)
                if col_sq.button("Via Security Question", use_container_width=True):
                    if not email:
                        st.error("⚠️ Please enter your email address.")
                    else:
                        with get_db() as c:
                            r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                        if r:
                            st.session_state.reset_email = email
                            st.session_state.security_question = r[0]
                            st.session_state.reset_mode = "sq"
                            st.rerun()
                        else:
                            st.error("❌ Email not found.")

                if col_otp.button("Via OTP", use_container_width=True):
                    if not email:
                        st.error("⚠️ Please enter your email address.")
                    else:
                        with get_db() as c:
                            r = c.execute("SELECT id FROM users WHERE email=?", (email,)).fetchone()
                        if not r:
                            st.error("❌ Email not found.")
                        else:
                            otp = generate_otp()
                            if send_otp(email, otp):
                                st.session_state.reset_email = email
                                st.session_state.reset_mode = "otp"
                                st.session_state.otp = otp
                                st.session_state.otp_sent_at = datetime.datetime.utcnow()
                                st.rerun()
                            else:
                                st.error("❌ Could not send OTP. Check EMAIL_ADDRESS/EMAIL_PASSWORD secrets.")
            else:
                if st.session_state.reset_mode == "sq":
                    st.info(f"❓ **Security Question:** {st.session_state.security_question}")
                    ans = st.text_input("Your answer").lower().strip()
                    npw = st.text_input("New password", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    st.caption(PASSWORD_HINT)
                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button("Reset Password →", use_container_width=True):
                        if not ans or not npw or not confirm_npw:
                            st.error("⚠️ Please fill all fields.")
                        elif not is_valid_password(npw):
                            st.error(f"❌ Weak password. {PASSWORD_HINT}")
                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c:
                                r = c.execute(
                                    "SELECT security_answer_hash FROM users WHERE email=?",
                                    (st.session_state.reset_email,)
                                ).fetchone()
                            if r and verify_password(ans, r[0]):
                                with get_db() as c:
                                    c.execute(
                                        "UPDATE users SET password_hash=? WHERE email=?",
                                        (hash_password(npw), st.session_state.reset_email)
                                    )
                                st.success("✅ Password updated successfully!")
                                st.session_state.reset_email = None
                                st.session_state.reset_mode = None
                                st.session_state.page = "Login"
                                st.rerun()
                            else:
                                st.error("❌ Incorrect security answer.")

                elif st.session_state.reset_mode == "otp":
                    expired = (
                        st.session_state.otp_sent_at
                        and (datetime.datetime.utcnow() - st.session_state.otp_sent_at).total_seconds() > 300
                    )
                    if expired:
                        st.warning("⏰ OTP expired. Please go back and request a new one.")
                    else:
                        st.info(f"📧 A 6-digit code was sent to **{st.session_state.reset_email}**")

                    otp_in = st.text_input("Enter OTP", max_chars=6)
                    npw = st.text_input("New password", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    st.caption(PASSWORD_HINT)
                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button("Verify & Reset Password →", use_container_width=True):
                        if not otp_in or not npw or not confirm_npw:
                            st.error("⚠️ Please fill all fields.")
                        elif expired:
                            st.error("❌ OTP expired.")
                        elif otp_in != st.session_state.otp:
                            st.error("❌ Incorrect OTP.")
                        elif not is_valid_password(npw):
                            st.error(f"❌ Weak password. {PASSWORD_HINT}")
                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c:
                                c.execute(
                                    "UPDATE users SET password_hash=? WHERE email=?",
                                    (hash_password(npw), st.session_state.reset_email)
                                )
                            st.success("✅ Password updated successfully!")
                            st.session_state.reset_email = None
                            st.session_state.reset_mode = None
                            st.session_state.otp = None
                            st.session_state.otp_sent_at = None
                            st.session_state.page = "Login"
                            st.rerun()

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                st.session_state.otp = None
                st.session_state.otp_sent_at = None
                navigate("Login")

# =====================================================
# DASHBOARDS
# =====================================================

else:
    payload = current_user()
    if payload is None:
        st.stop()

    email = payload["email"]

    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{TEXT};">Infosys Portal</div>
            <div style="font-size:11px;color:#64748b;">{"Admin Panel" if st.session_state.is_admin else "Intelligent Analytics"}</div>
        </div><hr>
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Logout"] if st.session_state.is_admin else ["Dashboard", "Analytics", "Reports", "Logout"]
        icons = ["house", "box-arrow-right"] if st.session_state.is_admin else ["house", "graph-up", "file-text", "box-arrow-right"]
        menu = option_menu(None, opts, icons=icons,
                            styles={"container": {"background-color": "#EAF4FF"},
                                    "nav-link-selected": {"background-color": PRIMARY, "color": "white"}})
        if menu == "Logout":
            logout_user()

    if st.session_state.is_admin:
        st.markdown(f"""
        <div style="background:{TEXT};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{ACCENT} !important;margin:0;">⚡ Infosys Portal</h1>
            <div style="color:#cbd5e1;font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{PRIMARY};padding:8px 18px;border-radius:30px;font-weight:700;color:white;">🛡️ Admin</div>
        </div>
        """, unsafe_allow_html=True)

        with get_db() as c:
            rows = c.execute("SELECT username, email FROM users ORDER BY id DESC").fetchall()

        st.markdown(f'<div class="card"><h3 style="margin-top:0;">👥 Registered Users ({len(rows)})</h3></div>', unsafe_allow_html=True)

        if rows:
            st.dataframe(
                {"Username": [r[0] for r in rows], "Email": [r[1] for r in rows]},
                use_container_width=True, hide_index=True
            )
        else:
            st.info("No registered users yet.")

    else:
        with get_db() as c:
            uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

        st.markdown(f"""
        <div style="background:{TEXT};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{ACCENT} !important;margin:0;">⚡ Infosys Portal</h1>
            <div style="color:#cbd5e1;font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{PRIMARY};padding:8px 18px;border-radius:30px;font-weight:700;color:white;">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f'<div class="card"><h2 style="margin-top:0;">Welcome, {uname} 👋</h2><p style="color:#64748b;">You are logged in.</p></div>', unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [
            (c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
            (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")
        ]:
            col.markdown(f"""
            <div class="card" style="text-align:center;margin-top:16px;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{TEXT};">{val}</div>
                <div style="color:#64748b;font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(
            mode="gauge+number", value=92,
            title={"text": "System Health Index", "font": {"color": TEXT, "size": 14}},
            gauge={"axis": {"range": [0, 100]}, "bar": {"color": PRIMARY}, "bgcolor": "#EAF4FF", "borderwidth": 1, "bordercolor": TEXT}
        ))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": TEXT}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

Writing app.py


In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# Pull secrets here (this cell has kernel access) and expose them to app.py
os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET")
os.environ["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS")
os.environ["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD")

NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
ngrok.set_auth_token(NGROK_TOKEN)

ngrok.kill()
!pkill -f streamlit

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit

🚀 Infosys Portal Live URL: https://nutshell-buffing-coke.ngrok-free.dev
